# RoSBERTa: base vs fine-tuned (3 ключевые метрики)

Сравнение `ai-forever/ru-en-RoSBERTa` (zero-shot) и дообученного варианта (`models/final/bi-encoder`, Augmented SBERT — CoSENTLoss на gold_train + silver). Метрики: Spearman ρ, Pearson r, MRR.

In [1]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали из подпапки
import os
from pathlib import Path
_p = Path.cwd()
while _p.name != 'thesis' and _p.parent != _p:
    _p = _p.parent
if _p.name == 'thesis':
    os.chdir(_p)
print('CWD:', Path.cwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
import warnings
warnings.filterwarnings('ignore')

import json
import html
import time
import gc
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
from scipy import stats
from IPython.display import HTML, display

# sentence-transformers 5.x ↔ transformers 4.57 совместимость
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import lancedb

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')


DEVICE: cpu


## Метрики

Оцениваем bi-encoder'ы по трём ключевым показателям на `golden_eval.parquet` (1598 пар с gold-скорами релевантности):

1. **Spearman ρ** — ранговая корреляция gold и cosine. Главная метрика: правильно ли модель упорядочивает пары по степени релевантности.
2. **Pearson r** — линейная корреляция. Показывает калибровку: ложатся ли точки `(gold, cos)` на прямую.
3. **MRR** — для каждого `desc_i` ищется свой `post_i` среди всех 1598 постов; средний обратный ранг. Прокси для retrieval-качества без отдельной разметки «правильных» постов.

Ниже — визуальная таблица top-5 на реальном индексе 50k. Это **не метрика**, а качественный контроль: на неё не опираемся при оценке качества.

In [3]:
# ==================== МОДЕЛИ ДЛЯ СРАВНЕНИЯ ====================
MODELS = [
    {
        'key':          'rosberta_base',
        'display_name': 'RoSBERTa base',
        'model_path':   'ai-forever/ru-en-RoSBERTa',
        'table_name':   'rosberta-base-50k',
        'doc_prefix':   '',
        'query_prefix': '',
        'color':        '#cfe2ff',
    },
    {
        'key':          'rosberta_ft',
        'display_name': 'RoSBERTa fine-tuned',
        'model_path':   'models/final/bi-encoder',
        'table_name':   'rosberta-fine-tuned-50k',
        'doc_prefix':   '',
        'query_prefix': '',
        'color':        '#9ec5fe',
    },
]


In [4]:
# --------- общие параметры ---------
LANCEDB_PATH    = './lancedb_store'
EVAL_PARQUET    = 'data/golden/golden_eval.parquet'
GT_POSTS_JSON   = 'ground_truth_posts.json'
GT_PAIRS_JSON   = 'ground_truth_pairs.json'
TOP_K_VISUAL    = 5     # сколько постов показать в визуальной таблице
BATCH_SIZE      = 64    # для encode на golden_eval (1598 строк)

print(f'Будет сравниваться моделей: {len(MODELS)}')
for m in MODELS:
    print(f"  - {m['display_name']:30s} ({m['table_name']})")


Будет сравниваться моделей: 2
  - RoSBERTa base                  (rosberta-base-50k)
  - RoSBERTa fine-tuned            (rosberta-fine-tuned-50k)


In [5]:
# Проверяем, что все нужные LanceDB-таблицы существуют (нужно только для визуала top-5)
db = lancedb.connect(LANCEDB_PATH)
available = set(db.table_names())
missing = [m for m in MODELS if m['table_name'] not in available]
if missing:
    msg = '\n'.join(f"  - {m['display_name']}: нет таблицы {m['table_name']}" for m in missing)
    raise RuntimeError(
        f'В {LANCEDB_PATH} отсутствуют таблицы для следующих моделей:\n{msg}\n\n'
        f'Сначала прогони thesis/db/create-all-dbs.ipynb для нужных моделей, '
        f'либо закомментируй их в MODELS выше.'
    )
print(f'Все {len(MODELS)} таблиц на месте.')

# Заодно проверяем golden_eval.parquet и GT-файлы (GT нужны только для визуала)
for path in [EVAL_PARQUET, GT_POSTS_JSON, GT_PAIRS_JSON]:
    assert os.path.exists(path), f'Нет файла: {path}'
print('golden_eval.parquet, ground_truth_*.json — на месте.')


Все 2 таблиц на месте.
golden_eval.parquet, ground_truth_*.json — на месте.


In [6]:
# Загружаем golden_eval.parquet (для метрик) и ground_truth_pairs.json (для визуала)
from datasets import load_dataset

eval_ds = load_dataset('parquet', data_files=EVAL_PARQUET, split='train')
descriptions = list(eval_ds['product_desc'])
posts        = list(eval_ds['post_text'])
gold_scores  = np.array(eval_ds['score'], dtype=float)
print(f'golden_eval: {len(descriptions):,} пар')

with open(GT_POSTS_JSON, encoding='utf-8') as f:
    gt_posts = json.load(f)
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    gt_pairs = json.load(f)
post_text_by_num = {p['post_id']: p['text'] for p in gt_posts}
print(f'GT (для визуала): {len(gt_pairs)} пар, {len(gt_posts)} эталонных постов')


golden_eval: 1,598 пар
GT (для визуала): 19 пар, 10 эталонных постов


In [7]:
# Функция: считает три ключевые метрики bi-encoder'а на golden_eval.parquet
# 1) Spearman ρ  — ранговая корреляция gold и cosine (главная — качество ранжирования)
# 2) Pearson r   — линейная корреляция gold и cosine (калибровка)
# 3) MRR         — retrieval: для каждого desc_i ищется post_i среди всех 1598 постов
def compute_metrics(model, model_cfg, descriptions, posts, gold_scores):
    desc_in = [model_cfg['query_prefix'] + d for d in descriptions] if model_cfg['query_prefix'] else descriptions
    post_in = [model_cfg['doc_prefix']   + p for p in posts]        if model_cfg['doc_prefix']   else posts

    desc_embs = model.encode(desc_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)
    post_embs = model.encode(post_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)

    # Корреляции на парах (i, i)
    pair_scores = (desc_embs * post_embs).sum(dim=1).cpu().numpy()
    spearman_r, _ = stats.spearmanr(gold_scores, pair_scores)
    pearson_r, _  = stats.pearsonr(gold_scores, pair_scores)

    # MRR: для каждого desc_i — ранг post_i среди 1598 кандидатов (пропускаем пары с gold<=0)
    sim_matrix = cos_sim(desc_embs, post_embs).cpu().numpy()
    rrs = []
    for i in range(len(descriptions)):
        if gold_scores[i] <= 0:
            continue
        order = np.argsort(-sim_matrix[i])
        rank = int(np.where(order == i)[0][0]) + 1
        rrs.append(1.0 / rank)

    return {
        'spearman': spearman_r,
        'pearson':  pearson_r,
        'mrr':      float(np.mean(rrs)) if rrs else 0.0,
        'eval_pairs': len(rrs),
    }


In [8]:
# Главный цикл: загружаем модели, считаем метрики
results = OrderedDict()
loaded_models = OrderedDict()  # нужны ниже для визуала

for cfg in MODELS:
    print(f"\n--- {cfg['display_name']} ({cfg['model_path']}) ---")
    t0 = time.time()
    model = SentenceTransformer(cfg['model_path'], device=DEVICE)
    print(f'  загружена за {time.time()-t0:.0f}с, dim={model.get_sentence_embedding_dimension()}')

    t0 = time.time()
    m = compute_metrics(model, cfg, descriptions, posts, gold_scores)
    print(f'  метрики посчитаны за {time.time()-t0:.0f}с')
    print(f'    Spearman={m["spearman"]:.4f}  Pearson={m["pearson"]:.4f}  MRR={m["mrr"]:.4f}')

    results[cfg['key']] = {'cfg': cfg, 'metrics': m}
    loaded_models[cfg['key']] = model

print('\nГотово.')



--- RoSBERTa base (ai-forever/ru-en-RoSBERTa) ---


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  загружена за 5с, dim=1024
  метрики посчитаны за 718с
    Spearman=0.1262  Pearson=0.2959  MRR=0.1030

--- RoSBERTa fine-tuned (models/final/bi-encoder) ---
  загружена за 2с, dim=384
  метрики посчитаны за 59с
    Spearman=0.3934  Pearson=0.3951  MRR=0.0572

Готово.


## Сводная таблица метрик

In [9]:
# Сводная таблица: Spearman, Pearson, MRR
rows = []
for key, r in results.items():
    m = r['metrics']
    rows.append({
        'Модель':     r['cfg']['display_name'],
        'Spearman ρ': round(m['spearman'], 4),
        'Pearson r':  round(m['pearson'],  4),
        'MRR':        round(m['mrr'],      4),
    })
df = pd.DataFrame(rows)
eval_n = next(iter(results.values()))['metrics']['eval_pairs']
print(f'Метрики на golden_eval.parquet ({eval_n} пар с gold>0)')
display(df.style.background_gradient(cmap='YlGn', subset=['Spearman ρ', 'Pearson r', 'MRR']))


Метрики на golden_eval.parquet (1034 пар с gold>0)


,Модель,Spearman ρ,Pearson r,MRR
0,RoSBERTa base,0.126200,0.295900,0.103000
1,RoSBERTa fine-tuned,0.393400,0.395100,0.057200


## Визуал: top-5 по каждой модели для каждой GT-пары (качественный контроль)

In [10]:
# Вспомогательная функция для визуала: экранирование + рендер одной пары
def esc(s):
    return html.escape(str(s)).replace('\n', '<br>')


def render_pair_topk(idx, total, pair, target_text, model_topk, top_k=5):
    target_norm = target_text.strip()

    col_headers = ''.join(
        f'<th style="padding:10px;text-align:left;background:{cfg["color"]};'
        f'border-bottom:2px solid #495057;color:#000;font-weight:bold;'
        f'border-right:1px solid #adb5bd;width:{round(100/len(model_topk), 2)}%;">'
        f'{esc(cfg["display_name"])}</th>'
        for cfg in (results[k]['cfg'] for k in model_topk.keys())
    )

    body = ''
    for rank in range(1, top_k + 1):
        cells = ''
        for key, rows in model_topk.items():
            if rank > len(rows):
                cells += '<td style="padding:10px;color:#000;border-right:1px solid #e9ecef;border-bottom:1px solid #e9ecef;vertical-align:top;">—</td>'
                continue
            r = rows[rank - 1]
            is_target = (r['text'].strip() == target_norm)
            bg = '#d4edda' if is_target else '#ffffff'
            border = '4px solid #28a745' if is_target else 'none'
            star = ' ★' if is_target else ''
            text_full = r['text'].replace('\n', ' ')
            cells += (
                f'<td style="padding:10px;color:#000;background:{bg};'
                f'border-left:{border};border-right:1px solid #e9ecef;'
                f'border-bottom:1px solid #e9ecef;vertical-align:top;font-size:12px;">'
                f'<div style="font-weight:bold;font-size:11px;margin-bottom:4px;color:#495057;">'
                f'#{rank}{star} · @{esc(r["channel"])} · {esc(r.get("category",""))}</div>'
                f'<div style="line-height:1.4;color:#000;">{esc(text_full)}</div>'
                f'</td>'
            )
        body += f'<tr>{cells}</tr>'

    return HTML(f'''
    <div style="border:2px solid #495057;border-radius:8px;margin:32px 0;
                background:#ffffff;font-family:system-ui,sans-serif;overflow:hidden;color:#000;">
        <div style="background:#e9ecef;padding:14px 20px;border-bottom:1px solid #ced4da;">
            <div style="font-size:17px;font-weight:bold;color:#000;">
                Пара {idx}/{total} · GT post #{pair["post_num"]}
            </div>
            <div style="font-size:13px;color:#495057;margin-top:4px;">
                {esc(pair.get("imt_name",""))} · {esc(pair.get("subj_name",""))}
            </div>
        </div>
        <div style="padding:14px 20px;background:#e7f3ff;border-bottom:1px solid #cfe2ff;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Запрос (описание товара)</div>
            <div style="color:#000;font-size:13px;line-height:1.5;">{esc(pair["description"])}</div>
        </div>
        <div style="padding:14px 20px;background:#d4edda;border-bottom:1px solid #c3e6cb;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Целевой пост (должен попасть в top-{top_k})</div>
            <div style="color:#000;font-size:13px;line-height:1.5;white-space:pre-wrap;">{esc(target_text)}</div>
        </div>
        <table style="width:100%;border-collapse:collapse;background:#ffffff;table-layout:fixed;">
            <thead><tr>{col_headers}</tr></thead>
            <tbody>{body}</tbody>
        </table>
    </div>
    ''')


In [11]:
# ============================================================
# ВИЗУАЛ: для каждой из GT пар — топ-5 по каждой модели
# Это качественный контроль, НЕ метрика. Нужен чтобы глазами увидеть,
# что модели вытаскивают осмысленные посты (даже если помеченный как
# 'правильный' пост не всегда оказывается в топ-5).
# ============================================================
for idx, pair in enumerate(gt_pairs, 1):
    target_text = post_text_by_num[pair['post_num']]
    model_topk = OrderedDict()
    for cfg in MODELS:
        model = loaded_models[cfg['key']]
        table = db.open_table(cfg['table_name'])
        q_in = (cfg['query_prefix'] + pair['description']) if cfg['query_prefix'] else pair['description']
        qvec = model.encode([q_in], normalize_embeddings=True)[0].tolist()
        rows = (table.search(qvec, query_type='vector')
                     .limit(TOP_K_VISUAL)
                     .select(['text', 'channel', 'category'])
                     .to_list())
        model_topk[cfg['key']] = rows
    display(render_pair_topk(idx, len(gt_pairs), pair, target_text, model_topk, top_k=TOP_K_VISUAL))


RoSBERTa base,RoSBERTa fine-tuned
"#1 · @Mirror_ofthe_soul · БлогиГидрофильное гель - масло от MIXITС ним макияж любой стойкости смывается за 2 минуты!Кожа абсолютно чистая без следов косметики,поры чистые, не вызывает воспалений, приятный аромат и не остается пленкой на глазах. Ищем нa:WB арт. 31299196 #длялица","#1 · @plotnik_remont · Интерьер и строительствоА вы знаете из чего состоит ламинат? Ламинат состоит из 5 различных слоев, которые спрессовываются между собой под высоким давлением: 1 Защитный слой из прочной смоляной пленки; 2 Декоративный слой, выполняемый, как правило, из бумаги или мебельной фольги, рисунок которой имитирует природный материал; 3 Пленка, увеличивающая влагостойкость, расположенная в середеине этого ламинатного ""пирога""; 4 Основной слой, плита HDF, панель ДВП или ДСП из древесно-волокнистой плиты высокой степени плотности, полученной методом горячего прессования; 5 Слой из влагостойкой бумаги или пластика, пропитанный меламиновой смолой, придающий панелям жесткость и защищающий пол от влаги."
"#2 · @zagorod_live · Интерьер и строительствоСУПЕР СРЕДСТВО УДАЛЯЕТ ЧЕРНУЮ ПЛЕСЕНЬ и ГРИБОК за 30 минут Фунгицид в составе обладает дезинфицирующим и отбеливающим эффектом для борьбы с плесенью, грибком и бактериями на различных поверхностях: керамических, пластиковых, бетонных, деревянных и окрашенных. Отличное средство для чистки грибка в стиральной машине. Средство для удаления от плесени и грибка идеально для применения в местах повышенной влажности: от плесени в ванной и на кухне. Цена: 549₽Артикул: 150118120","#2 · @look_decor · Интерьер и строительствоПроцесс замены старого герметика в ванной.1. Срезаем старый герметик.2. Убираем все остатки канцелярским ножом.3. Обрабатываем средством от плесени.4. Обезжириваем.5. Заклеиваем поверхность малярным скотчем, оставляя только сам шов.6. Заполняем шов силиконовым герметиком (так как ванна акриловая, она немного подвижна, силиконовый герметик лучше подходит и не будет трескаться).7. Для лучшего скольжения необходимо побрызгать герметик мыльным раствором.9. И провести карточкой, чтобы сделать шов ровным. Готово!"
"#3 · @posudakukmara · Еда и кулинарияКрасота, стиль, долговечность – эти эпитеты идеально подходят для описания нашей новой коллекции текстиля ""Правила кухни"". Комплект выполнен из плотного материала - рогожки, стойкого к истиранию. Он не абсорбирует запахи, мгновенно поглощает жидкости и быстро сохнет, не линяет после стирки и легко гладится.Казалось бы, этот комплект уже идеален, да? А если мы скажем, что с ним прекрасно сочетается набор кухонных принадлежностей из силикона, Ваша любовь к готовке станет ещё сильнее.Вот, теперь Вы непременно их хотите!? В коллекциях 3 цвета - подбирайте самый подходящий вариант для себя или в подарок на сайте","#3 · @zagorod_live · Интерьер и строительствоСУПЕР СРЕДСТВО УДАЛЯЕТ ЧЕРНУЮ ПЛЕСЕНЬ и ГРИБОК за 30 минут Фунгицид в составе обладает дезинфицирующим и отбеливающим эффектом для борьбы с плесенью, грибком и бактериями на различных поверхностях: керамических, пластиковых, бетонных, деревянных и окрашенных. Отличное средство для чистки грибка в стиральной машине. Средство для удаления от плесени и грибка идеально для применения в местах повышенной влажности: от плесени в ванной и на кухне. Цена: 549₽Артикул: 150118120"
"#4 · @skidky7 · ПродажиСохрани на WB # товары для дома Цена : 668 рублей — это 100 стирок Уже можно не переплачивать за Tide и Ariel 4.7 Только честные отзывы Отстирает на ура, жирные пятна, выполаскивание в холодной воде, парфюмированная добавка,которая заменит кондиционер, оставляя слегка уловимый аромат, без химозного запаха, подходит для детей Еще, этот порошок используют в сети отелей, но под другим брендом Смотреть тут","#4 · @remontiruyrukami · Интерьер и строительствоВ стыках между плиткой и ламинатом/массивом/паркетом кладут либо пробку, либо молдинг (выпуклая планка). Обязательно нужно знать монтажный размер молдинга, если вы собираетесь его использовать."
"#5 · @sweethometop · Интер

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @sweethometop · Интерьер и строительствоСовременные интерьеры тяготеют к минимализму: меньше мебели, 2-3 цвета в палитре комнаты, ограниченное количество аксессуаров.Популярный вариант, который придаст вашему жилью свежий вид - это отсутствие обоев.Для того, чтобы воплотить этот тренд вам потребуются лишь ровные стены и краска.При выборе цвета обратите внимание на то, чтобы пигмент был насыщенным. Такая отделка будет придавать комнате глубину.Советуем обратить внимание на эти цвета: ⁃ белый ⁃ черный ⁃ коричневый ⁃ фиолетовый ⁃ желтый ⁃ синий ⁃ зелёный ⁃ красныйЕсть еще два несомненных плюса крашеных стен: экологичность и удобство в уходе.Sweet Home","#1 · @plotnik_remont · Интерьер и строительствоА вы знаете из чего состоит ламинат? Ламинат состоит из 5 различных слоев, которые спрессовываются между собой под высоким давлением: 1 Защитный слой из прочной смоляной пленки; 2 Декоративный слой, выполняемый, как правило, из бумаги или мебельной фольги, рисунок которой имитирует природный материал; 3 Пленка, увеличивающая влагостойкость, расположенная в середеине этого ламинатного ""пирога""; 4 Основной слой, плита HDF, панель ДВП или ДСП из древесно-волокнистой плиты высокой степени плотности, полученной методом горячего прессования; 5 Слой из влагостойкой бумаги или пластика, пропитанный меламиновой смолой, придающий панелям жесткость и защищающий пол от влаги."
"#2 · @decor_kv · Интерьер и строительствоДизайн интерьера 3 полезных мелочей для дома.Вещи, которые облегчат вашу бытовую жизнь! Легкая и тонкая стремянка Держатель для швабры Органайзер для капсул для посудомоечной машиныДекор Квартиры","#2 · @shelnat · Интерьер и строительствоКакая-то невероятно и безупречно прекрасная кухня от L’Ottocento, построенная на контрастах теплого дерева французских буазери с флером классики и уюта, и холодного металла, который придает гигиеничность и современность. Очень красиво. Впрочем, контраст классики и современности — это стиль бренда, но тут все выполнено особенно безукоризненно. Хочется затаить дыхание от красоты.*Вообще, на итальянских кухнях стало очень много металла: на фасадах, столешницах, фартуках. Это несомненно тренд. При этом не весь металл издает неприятные звуки, не весь металл прохладен, не весь металл подвержен отпечаткам и царапинам. Уже существует много материалов, сильно напоминающих металл, но лишенных его недостатков."
"#3 · @rem_shkola · Интерьер и строительствоСтеклянные перегородки - простой способ зонировать помещение, сохранить свет и сделать квартиру стильной!Хитрость в современном дизайне и способности перегородок пропускать солнечный свет. Они визуально расширяют пространство, при этом зонируют его и делают помещение более функциональным Шумоизоляция позволит отгородиться от животных и детей без потери квадратных метров. Стекло может быть прозрачным, матовым или рельефным, цвет и форма перегородок — под ваш интерьер и зону, где они устанавливаются.Школа ремонта","#3 · @menberloga · Интерьер и строительство⁣Пресс-вайм для склеивания дерева.Профиль 50х25. Поскольку он тонкостенный, ввариваются внутрь втулки распорные из полдюймовой трубы. Шпилька 12-ю разрезается на четыре части, к одной стороне каждой из частей привариваются ручки из трубы. Предварительно надрезается торец трубы крестом и подвальцевывается, и просверливается она для воротка. Башмаки упорные свариваются из двух уголков. Гайка М12, навертывается внутри башмака на шпильку, просверливается вместе со шпилькой и зашплинтована гвоздем для вагонки.Чтобы башмак держался на шпильке - просверливается под винт в потай М6 опорная плоскость и сама шпильку, нарезается в ней резьба. Все шплинтуется гвоздями для вагонки. Все болты и гайки М10, кроме винтовой пары (М12). Саму винтовая пара сделана из удлиненных гаек М12, приваривается к ним по паре пластин с отверстиями и, соответственно, обрезков шпильки М12. В заднем шарнире втулка не приварена (между пластинами)."
"#4 · @rem_shkola · Интерьер и строительствоСтеклянные

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @logopedia · Семья и детиЗАПОМИНАЮ БУКВЫ Колесникова Е. В. Книга-тетрадь способствует:- формированию зрительного образа буквы, - закреплению связи звука и буквы, - развитию графических навыков с целью подготовки руки ребенка к письму. Адресуется педагогам, гувернерам, родителям, а самое главное - детям, которым предстоит выполнять задания этой книги.","#1 · @uchim_sami · Семья и детиРебёнок не хочет читать? Значит будем с ним играть! Ведущий вид деятельности дошкольника - игра. Именно через неё ребёнок должен познавать мир, обучаться и развиваться. Сюжетные игры вызывают огромный интерес у ребёнка. Поэтому я создала сказки в стихах для обучения детей чтению, для комплексного развития и качественной подготовки к школе. Летом детям не хочется заниматься, но участвовать в увлекательных приключениях, рисовать, раскрашивать, проходить лабиринты, приклеивать картинки и т.д.прямо на даче на свежем воздухе будет с радостью каждый ребёнок! Знакомство и отработка навыка чтения с каждой буквой будет проходить в виде увлекательной сказки в стихах. По ходу развития её сюжета ребёнок будет помогать героям, проходить через препятствия, путешествовать вместе с ними, а самое главное: знакомиться с буквами, учиться читать и развиваться.Дети ждут с нетерпением каждую следующую сказку! Они твердо усваивают - учиться интересно! Это и есть залог успеха!Переходите в интернет-магазин и покупайте сказки, которые сделают детство счастливым, а взрослую жизнь - успешной!"
"#2 · @facktoriym · ПозновательноеЯркий и необычный подарок, который понравится абсолютно всем Развивает воображение, мелкую моторику и усидчивость! Увлекает надолго как взрослого, так и ребенка. Готовая работа классно впишется в любой интерьер ЦЕНА: 815 руб.","#2 · @kudri_v_oblaka · БлогиНОВОГОДНИЙ ДЕТСКИЙ АДВЕНТ *бесплатно*Что вас ждет:• 4 добрые, уютные новогодние СКАЗКИ – по одной на каждую неделю декабря• 4 интересных, необычных, но простых МАСТЕР-КЛАССА для детей (и не только)) в формате видео с подробной инструкцией• 4 АУДИО записи сказок в исполнении вашей кудрявой Нади, мамы двух кудрявых мальчиков, и поэтому знающей толк в сказках Каждый вторник объявляю сказочным!Весь декабрь по вторникам вас ждет БЕСПЛАТНАЯ сказка, которую вы можете прочитать сами своему малышу, а можете включить аудио запись. Герои сказки пригласят вас и вашего ребенка выполнить интересное задание, которое мы с Даней будем выполнять вместе с вами. СКАЗКИ подарят вам и вашему малышу новогоднее настроение и ощущение приближения праздника. Все истории связаны по сюжету – это сказка с продолжением, когда дети с интересом ждут: а что будет в дальше? АУДИО сказка спасет ваш вечер и поможет уснуть вашему малышу. МАСТЕР-КЛАССЫ подобраны специально для ребят-дошколят. Они простые и не требуют приобретения дорогих материалов. Все необходимое найдется дома. ВИДЕО мастер-класса сделает процесс изготовления поделки максимально понятным и простым."
"#3 · @facktoriym · ПозновательноеЯркий и необычный подарок, который понравится абсолютно всем Развивает воображение, мелкую моторику и усидчивость! Увлекает надолго как взрослого, так и ребенка. Готовая работа классно впишется в любой интерьер ЦЕНА: 815 руб.","#3 · @uchim_sami · Семья и детиВаш ребёнок знает секретики буквы Ы?Эта буква не простая и требует особого внимания. Деткам бывает сложно её запомнить. Но только не тем, которые занимаются по моим книгам! СКАЧАЙТЕ БЕСПЛАТНО ПРОДОЛЖЕНИЕ ПРОПИСИ!Ваш ребёнок узнает секреты буквы Ы, научится её писать. А так же научится писать слоги, слова и предложения с этой буквой. Для лучшего усвоения буквы и доведения навыка чтения с этой буквой до автоматизма, предлагаю Вам использовать пропись параллельно со сказками в стихах для обучения чтению. Ваш ребёнок испытывает сложности чтения с какой-либо буквой? Просто купите сказку с этой буквой в моем интернет-магазине в бумажном или электронном виде и забудьте про сложности! Чтобы получить продолжение прописи: подпишись на мой инстаграм аккаунт пос

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @Fastrecepts · Еда и кулинарияПридумать поделку в школу или детский садик теперь не проблема! ДЕТСКИЕ ПОДЕЛКИ — это огромная коллекция идей для детских поделок. На любой вкус и запас времени, от самых простых на скорую руку, до высокохудожественных. Собраны по темам - удобно!Сделать их можно буквально за 5 минут, а радости у детей хоть отбавляй!Подписывайтесь и пользуйтесь:","#1 · @mamochka_ryadom · Семья и детиЗИМУШКА-ЗИМААппликация из ваты Аппликация — интереснейший вид творчества, особенно для детей, которые любят создавать детские поделки своими руками. А так как за окном зима, то и аппликация должна быть мягкой и пушистой, как зимний снег. За основу возьмите вату: она как раз напоминает снег. Вата — хороший материал для объёмной зимней аппликации и других поделок. Потребуются:• цветной картон• вата • клей• фломастер• блестки (по желанию) На бумаге фломастером рисуем дерево. Из кусочков ваты скатываем жгутики и приклеиваем их к ветвям дерева. На заднем плане из ватных жгутиков делаем холмы, фломастером рисуем деревья. Делаем снежную крону деревьям, рисуем кусты. Из ваты делаем облака на небе и крону кустам."
"#2 · @Fastrecepts · Еда и кулинарияПридумать поделку в школу или детский садик теперь не проблема! ДЕТСКИЕ ПОДЕЛКИ — это огромная коллекция идей для детских поделок. На любой вкус и запас времени, от самых простых на скорую руку, до высокохудожественных. Собраны по темам - удобно!Сделать их можно буквально за 5 минут, а радости у детей хоть отбавляй!Подписывайтесь и пользуйтесь:","#2 · @kudri_v_oblaka · БлогиНОВОГОДНИЙ ДЕТСКИЙ АДВЕНТ *бесплатно*Что вас ждет:• 4 добрые, уютные новогодние СКАЗКИ – по одной на каждую неделю декабря• 4 интересных, необычных, но простых МАСТЕР-КЛАССА для детей (и не только)) в формате видео с подробной инструкцией• 4 АУДИО записи сказок в исполнении вашей кудрявой Нади, мамы двух кудрявых мальчиков, и поэтому знающей толк в сказках Каждый вторник объявляю сказочным!Весь декабрь по вторникам вас ждет БЕСПЛАТНАЯ сказка, которую вы можете прочитать сами своему малышу, а можете включить аудио запись. Герои сказки пригласят вас и вашего ребенка выполнить интересное задание, которое мы с Даней будем выполнять вместе с вами. СКАЗКИ подарят вам и вашему малышу новогоднее настроение и ощущение приближения праздника. Все истории связаны по сюжету – это сказка с продолжением, когда дети с интересом ждут: а что будет в дальше? АУДИО сказка спасет ваш вечер и поможет уснуть вашему малышу. МАСТЕР-КЛАССЫ подобраны специально для ребят-дошколят. Они простые и не требуют приобретения дорогих материалов. Все необходимое найдется дома. ВИДЕО мастер-класса сделает процесс изготовления поделки максимально понятным и простым."
"#3 · @Fastrecepts · Еда и кулинарияПридумать поделку в школу или детский садик теперь не проблема! ДЕТСКИЕ ПОДЕЛКИ — это огромная коллекция идей для детских поделок. На любой вкус и запас времени, от самых простых на скорую руку, до высокохудожественных. Собраны по темам - удобно!Сделать их можно буквально за 5 минут, а радости у детей хоть отбавляй!Подписывайтесь и пользуйтесь:","#3 · @sonya_gda · Еда и кулинарияЗнаю, что многие из Вас мечтают научиться искусству оформления бранчей и фуршетов.Поэтому, в предверии ярких новогодних праздников, где застолья и подарки для родных и близких - самые главные хлопоты, я приглашаю Вас на мой ДВУХДНЕВНЫЙ ОНЛАЙН МАСТЕР-КЛАСС 7 и 8 ДЕКАБРЯ В первый день мы: Соберём бранч-боксы в эфире и поговорим о их разнообразии; Упакуем готовые боксы и разберём декор; Прямо в эфире мы попробуем продать наши шедевры и посмотрим, сколько можно заработать на одном бранч-боксе, особенно накануне праздников. На 2-ой день Я расскажу и покажу Вам, как грамотно и при этом стильно засервировать фуршет; Какие нюансы я учитываю и какие лайфхаки помогают избегать ошибок; Где искать первых клиентов и почему сегодня фуршет - это тренд на любом празднике.ДЛЯ ТЕХ, КТО ПРОВЕДЁТ С НАМИ ОБА ДНЯ МАСТЕР-КЛАССА ДО КОНЦА МЫ ПОДГОТОВИЛИ ВИДЕОУРОК 

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @leya_leyaaa · Мода и красотаПоследнее время моя активность снизилась, поэтому я срочно оформила заказ у уже любимых и проверенных Re-Feel и заказала для себя целых три коробки полезных стиков, то-есть витаминов, которые способствуют не только укреплению иммунитета, но и работают на влияние стресса, укрепляют мышцы, восполняют все нужные нам в организме функции! Давайте расскажу что же я приобрела и для чего это собственно нужноМорской коллаген с клубникой в стикахВсе мы знаем, что коллаген-это бустер для здоровья кожи, волос,костей и мышц, поэтому я срочно заказала именно эту коробочку чтобы значительно повысить свое здоровье и укрепить иммунитет Матча-латте Ее я взяла для того, чтобы усилить эффект сияющей кожи.В составе так же присутствует коллаген в формате пептидов, который усваивается быстрее и на 100% делает кожу более упругой, ногти крепкими а волосы блестящими Так же в составе присутствует пребиотик инулин, который поддерживает ЖКТ И фиолетовая коробочка, это у нас ассорти из матчи, кофе, какао и чая Матча с коллагеном, кофе с пребиотиком, пряный чай латте для иммунитета и антистресс-какое для расслабления Будем восстанавливать силы вкусно и полезно","#1 · @ppshhka · Здоровье и фитнесСтартовала БЕСПЛАТНАЯ ФЕДЕРАЛЬНАЯ ПРОГРАММА похудения и оздоровления населения! Программа направлена на людей:- с лишним весом и ожирением;- старше 35 лет;- с заболеваниями ЖКТ.В основе лежит метод МЕТАБОЛИЧЕСКОГО ПОХУДЕНИЯ.Суть метода в том, чтобы восстановить обмен веществ и вместе с этим:- сбросить лишний вес,- избавиться от проблем с ЖКТ,- нормализовать давление,- омолодить и оздоровить организм.Организатор федеральной программы – международный центр похудения Кристины Шереметьевой, в котором уже более 12000 человек избавились от лишнего веса.Вы можете бесплатно принять участие в 5-дневной программе похудения!вы получите:- пошаговый план похудения,- специальное метаболическое меню,- методики для избавления от тяги к сладкой и жирной пище.Вы начнете восстанавливать метаболизм и работу ЖКТ и терять лишний вес без голодовок – до 4 кг за время программы. Старт 26 ИЮНЯ . Чтобы стать участником, зарегистрируйтесь:"
"#2 · @vstalaipohla · Мода и красотаФантастическая четверка против целлюльных нападений:Ролл МФР — болезненный, но с помощью упражнений можно эффективно и быстро раскатать бугристые застоиАппликатор Кузнецова — самый простой из всех. Просто выйти распаренной из душа, намазаться антицеллюльным кремом и сесть на игольчатый коврик минут на 15FatSecret - классное приложение для подсчета калорий и контроля КБЖУ. Потому что это самый эффективный способ наладить питание и контролировать его, а без питания остальные пункты могут не помочьБанки для вакуумного массажа — самый трудозатратный, но очень эффективный. Делать по кокосовому маслу круговыми движениями, пока бедра не начнут гореть. Лучше брать крупные, мягкие банки, чтобы не налепить синяковЧто добавили бы или какой больше предпочитаете? Делитесь своими способами","#2 · @shkola_dietologov · Здоровье и фитнесБич нашего времени и как его победить? Речь про высокий холестерин. Верный спутник практически каждого человека, которому уже исполнилось 35-50 лет. Скажем честно: в борьбе с холестерином придется кое-чем пожертвовать Но все не так страшно! На самом деле, нормализация уровня холестерина и профилактика его повышения – это легко, интересно и вкусно. Докажем каждое слово на новом обучающем курсе УПДН «Холестерин под контролем»! Что вы узнаете на курсе:- Определение и функции холестерина: что это такое как влияет на наш организм?- Общие причины повышения холестерина в крови- Заболевания, связанные с нарушением обмена холестерина- Диaгностика лабораторная и инструментальная для выявления проблем с холестерином- БАД, витамины и минералы для поддержания оптимального уровня холестерина- Профилактика и немедикаментозное лечение повышенного холестеринаТакже вы получите рацион питания на 7 дней для нормализации уровня холестерина! Сейчас можно присое

...

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @favouriteWB24 · Мода и красотаС такой подушкой каждое утро будет добрым!Подушка с эффектом памяти Поможет расслабить мышцы шеи и спины и занять правильное положение во время сна. Чехол снимается, а два валика помогут подобрать комфортную высоту подушки под себя. Цена сейчас: 1869₽ Обычная цена: 7331₽ Успей заказать","#1 · @good_air1 · ТехнологииСУШКА ВОЛОС ЗА 2 мин. 50 сек. Тот самый нашумевший фен, о котором мечтая каждая девушка Фен с насадкой концентратором за считанные минуты высушит и сделает шикарную укладку волосам любой длины за максимально быстрое время! В комплекте 5 видов насадок: 1. Концентратор для целенаправленной укладки 2. Диффузор для четких локонов и волн 3. Насадка Flyaway для гладкой поверхности волос 4. Новая насадка Gentle Air для тонких волос и чувствительной кожи головы 5. Классическая насадка После сушки волос можно пользоваться плойкой и выпрямителем, так как фен не травмирует волосы, благодаря своей уникальной технологии!Очень быстро сушит волосы Отличный подарок маме, жене, девушке дочке, бабушке на любой повод (а лучше - без повода ). Цена - 9990₽Для заказа пишите нам"
#2 · @wb_ozon_sale_skidki · Мода и красотаПодушкаЦена: 656₽ (вместо 4 005₽)#товарыдо1000 Анатомическая подушка 50х70 сочетает в себе функциональность и комфорт. Она обладает антистрессовым эффектом и помогает справиться с храпом благодаря специальной технологии антихрапа. Создана из натуральных материалов: наполнитель - микроволокно и чехол из микрофибры. Изделие является гипоаллергенным и подходит даже для самой чувствительной кожи.Ссылка:,"#2 · @knitideas · Рукоделие​​Универсальная классная шапка из Hamelton Tweed 1от магазина пряжи hollywoolВо-первых, она двусторонняя: особая макушка из четырех клиньев хорошо смотрится как с лицевой, так и с изнаночной стороны. Во-вторых, шапка из Hamelton Tweed подойдет и мужчинам, и женщинам. Стильная и теплая! На спицы 4 мм набираем 88 п (плотное облегание на голову 53-54 см, либо больше на ваш размер - петли должны быть кратны 2). Замыкаем в круг.Вяжем резинкой 1 на 1 около 57 рядов или 25 см (в моем случае, но лучше мерить по своей голове). Убавки макушки:разделяем петли на 4 клина с учетом, что на каждую убавку уходит по 5 петель (убавка 2 п с наклоном влево, 1 изнаночная, убавка 2 п с наклоном вправо = 5 петель). В каждом клине убавляем по 2 петли, значит в одном ряду убавляем 8 петель. Убавки вяжем через ряд. Так вяжем до тех пор, пока на спицах не останется 8 петель (или около того, если у вас изначально другое количество петель). Их убавляем все по 2 вместе вправо. Оставшиеся 4 петли стянуть, заправить нить. Провести ВТО, высушить и носить с удовольствием"
#3 · @wbskyy · ПродажиПушистая косметичка Арт: 166056491 Цена: 383 рубля О товаре Легко помещается в женскую сумку. Отлично подойдет для путешествий. Качественная молния гарантирует надежное закрывание. Ссылка на товар,"#3 · @poizondelivery · Мода и красотаСандалии с носками. За или против Сандалии с носками могут показаться верхом безвкусицы, но в последнее время именно такое сочетание мы все чаще видим на знаменитостях и показах модных брендов. Санадли с носками долгое время были стилистическим табу и ассоциировались только с пенсионерами , которым нет дела до моды, главное — комфорт. Именно комфорт и был главной отличительно чертой такого сочетания, и как раз он помог легитимизировать стилистический прием.Особенно сочетание сандалий с носками понравилось главным адептам стритвира — рэперам Особенность носков в таких образах еще и в том, что их видно , а значит — они могут быть полноценной частью образа. Можно играть на контрастах, сочетая вещи разных фактур, как A$AP Rocky, — тогда шерстяные носки крупной вязки станут акцентом. А можно выбрать цветные носки по примеру Джастина Бибера, которые будут подходить к другой яркой вещи в наряде.Также носки в сочетании с сандалиями в летний сезон не доставляют излишнего дискомфорта и неудобства ноге - вам не будет жарко Обратная связь и оформление зак

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @wildberries2016 · Мода и красотаСумка шоппер стёганая Чтобы выделиться из толпы, показать свою индивидуальность, шопперы подойдут как нельзя лучше Этот аксессуар который должен быть в гардеробе каждой. Повседневная,вместительная женская сумка на плечо выполненная из стёганой водонепроницаемой ткани. Отличный подарок девушке, женщине, бабушке, дочке, маме, жене, сестре, коллеге, подруге 8 марта, новый год, день влюбленных, на день рождения, просто так без повода.","#1 · @kaleydoskopWB · Мода и красотаШоппер на молнии Цена: 995₽ 2 200₽ (-55%)Милая сумка шоппер идеальна на каждый день. Сумка выполнена из ткани вельвета. Мягкая и вместительная. Прочная модель имеет плотную текстуру, а также удобные отсеки с карманами внутри. Непромокаемая и водонепроницаемая подкладка защищает содержимое от влаги. Прямоугольная форма позволяет комфортно носить её на плече.Ссылочка на WB"
"#2 · @SumkiBijuteryotRuslana · Мода и красотаДорожная сумка-это идеальный выбор не только для любителей спорта.Её можно взять с собой не только на тренировку в зал или бассейн,но и в путешествие. Она идеальна для похода в спортзал или для поездки, в качестве ручной клади в самолёт.Непромокаемая сумка изготовлена из качественной водонепроницаемой ткани, что обеспечивает долговечность и защиту ваших вещей.У неё вместительное внутреннее отделение для хранения спортивной формы и сменной одежды, отдельный боковой карман для обуви, а также наружные и внутренние карманы для размещения аксессуаров . Большая сумка также оснащена удобными ручками и регулируемыми плечевым ремнём для комфортного ношения через плечо в дороге.Опт 500₽ роз 600₽Размер 45#27#20","#2 · @viadolorosa_vintage · Мода и красотаНовинка! Сумка Gucci PRINCY Выполнена из ткани с классическим узором бренда GG. Отделана натуральной кожей золотистого цвета. Фурнитура на сумке бледно золотистого оттенка.Имеет винтажные потертости на ручке и уголочках.Сумочка выглядит достаточно аккуратной и небольшой, но при этом очень вместительна: в основное отделение помещается кошелёк, ежедневник А5, косметичка и другое.Удобные ручки хорошо помещаются на плече.Имеет сертификат аутентификации Oskelly Размеры:- ширина 33 см- высота 16 см- глубина 14 смСтоимость 36.500 Сумочку можно приобрести с беспроцентной рассрочкой от 6.084₽/месВинтаж & ресейл Заказать через менеджера телеграм: #сумки_viadolorosa#gucci_viadolorosa"
"#3 · @SumkiBijuteryotRuslana · Мода и красотаДорожная сумка-это идеальный выбор не только для любителей спорта.Её можно взять с собой не только на тренировку в зал или бассейн,но и в путешествие. Она идеальна для похода в спортзал или для поездки, в качестве ручной клади в самолёт.Непромокаемая сумка изготовлена из качественной водонепроницаемой ткани, что обеспечивает долговечность и защиту ваших вещей.У неё вместительное внутреннее отделение для хранения спортивной формы и сменной одежды, отдельный боковой карман для обуви, а также наружные и внутренние карманы для размещения аксессуаров . Большая сумка также оснащена удобными ручками и регулируемыми плечевым ремнём для комфортного ношения через плечо в дороге.Опт 500₽ роз 600₽Размер 45#27#20","#3 · @antikvarananas · ПродажиСумкаЦена: 1 453₽ (вместо 4 000₽) 4.7Женская сумка 18х19см в стиле кросс-боди сделана из высококачественной экокожи. Сумка держит форму, её материал не трескается и не стирается со временем, она имеет три отделения внутри включая внутренний карман на молнии, застежкой для нее служит магнитный замок и молния. Также вместе с сумкой идут 2 разных ремешка - тонкий черный и широкий цветной. Доступна в двух цветах.Ссылка:"
"#4 · @wbnaxodkioz · БлогиСумка в стиле Prada, но без логотипа Полный аналог в экокоже, очень мягкая на ощупь! В комплекте широкий ремешок, ремешок-цепь. В носке на плече, через плечо, в руках Идеальное дополнение к любому образу Арт. 91037737","#4 · @antikvarananas · ПродажиСумкаЦена: 1 453₽ (вместо 4 000₽) 4.7Женская сумка 18х19см в стиле кросс-боди сделана из высококачественной эко

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @AndreyPitertsov · ПриродаПрошло уже два полноценных сезона, как я активно гоняю крупные приманки спиннинговым комплектом. Палочки в моих руках вы могли видеть разные: Хайрон до 140 или 200 г. Хелл Хаунд до 160... А вот мясорубка всегда одна - Твин 20 года 4000pg.Знаю, что многим интересно как она? Что с ней случилось после достаточно продолжительных нагрузок, не развалилась ли?К удивлению для многих скажу, что с ней все хорошо. Ход стал немного более грубым - это единственное что произошло, и это нормально. Больше и добавить нечего. Громыхать, урчать, скрипеть не начала.Так что практикой удалось подтвердить то, что ""мясорубочный"" комплект отлично подходит для больших приманок. Далее каждый сам решает на что ему ловить приятнее и комфортнее (мульт или мясорубка).","#1 · @ribalka_sovet · ПозновательноеТЕХАССКАЯ ОСНАСТКА — ИЗГОТОВЛЕНИЕ И ТЕХНИКА ЛОВЛИ Техасская оснастка относится к разряду абсолютных «незацепляек» и своим происхождением обязана одному из озер в штате Техас. Изначально оснастка применялась для ловли американского басса, но в силу своей универсальности и уловистости она завоевала популярность и у наших спиннингистов. В условиях наших водоемов Техасская оснастка с успехом применяется для ловли щуки, окуня, судака и даже язя. Что представляет собой Техасская оснастка и где применяется? Техасская оснастка устроена достаточно просто — офсетный крючок с насаженным на него силиконовым червем привязан к основной леске, на которой скользит грузило в виде пули. Между крючком и пулей устанавливается бусинка (бисер). Хотя ее наличие не всегда обязательно, но об этом позже."
"#2 · @rezatribu · ПозновательноеФинский пуукко с рукоятью из стабилизированной древесины. Сталь использовал 95х18.Такой нож станет идеальным помощником и верным спутником путешественников, охотников и рыбаков.","#2 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
"#3 · @rezatribu · ПозновательноеФинский пуукко с рукоятью из стабилизированной древесины. Сталь использовал 95х18.Такой нож станет идеальным помощником и верным спутником путешественников, охотников и рыбаков.","#3 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
"#4 · @ohota_ru · ПриродаРаз уж пошла дискуссия про ножи, добавлю своих пять копеек. У меня ножи были разные, один стоил 60 000 японский. Я их все подарил друзьям. Не зашли, то 

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum.","#1 · @ribalka_sovet · ПозновательноеТЕХАССКАЯ ОСНАСТКА — ИЗГОТОВЛЕНИЕ И ТЕХНИКА ЛОВЛИ Техасская оснастка относится к разряду абсолютных «незацепляек» и своим происхождением обязана одному из озер в штате Техас. Изначально оснастка применялась для ловли американского басса, но в силу своей универсальности и уловистости она завоевала популярность и у наших спиннингистов. В условиях наших водоемов Техасская оснастка с успехом применяется для ловли щуки, окуня, судака и даже язя. Что представляет собой Техасская оснастка и где применяется? Техасская оснастка устроена достаточно просто — офсетный крючок с насаженным на него силиконовым червем привязан к основной леске, на которой скользит грузило в виде пули. Между крючком и пулей устанавливается бусинка (бисер). Хотя ее наличие не всегда обязательно, но об этом позже."
"#2 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum.","#2 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
"#3 · @tovary_wildberriesss · Видео и фильмыНабор из 6-ти контейнеров из пищевого РЕТ пластика для хранения сыпучих продуктов. Помогут организовать хранение на кухне и экономят пространство, ставятся друг на друга.Идеально подойдут для хранения круп, крупы, муки, гречки, риса, макарон, орехов, конфет, печенья, кофе, чая, овощей, фруктов, а также для хранения продуктов в холодильнике. В набор входит 1 банка 1800 мл. + 1 банка 1300 мл+ 2 банки по 700 мл + 2 банки по 460 мл.Перейти к товару","#3 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
"#4 · @AndreyPitertsov · ПриродаПрошло уже два полноценных сезона, как я активно гоняю крупные приманки спиннинговым комплектом. Палочки в моих руках вы могли

RoSBERTa base,RoSBERTa fine-tuned
"#1 · @odezhdaf · Мода и красотаЗимняя женская панама - модная альтернатива шапкам в холодное время года. Отлично подойдет под любую верхнюю одежду, как под куртку, пуховик, так и под шубу с пальто. Меховая панама выглядит стильно и женственно. Женская утепленная панама регулируется по размеру.Черный: арт. 184512380Бежевый: арт. 184510793Белый: арт. 184509267 Заказать товар","#1 · @zendenofficial · ПродажиВозвращение классики: главный тренд весны для мужчинЭтой весной модные мужские тренды возвращаются к классике, и мир захватывает стиль Eclectic grandpa.Теперь мужчины могут оживить свой гардероб с помощью многослойных образов и оригинальных сочетаний. Центральные элементы в образе Eclectic grandpa – это рубашки с яркими принтами, теплые кардиганы и качественная, классическая обувь из натуральной кожи. Палитра цветов в этом тренде насыщена оттенками коричневого, добавляя образам утонченности и шарма.Хотим поделиться с вами подборкой обуви, которая идеально дополнит ваши образы в стиле Eclectic grandpa:1. Оксфорды2. Дерби3. Ботинки4. Броги5. ЛоферыДайте волю своей креативности и создавайте неповторимые образы с изысканной ноткой ретро.С любовью, ZENDEN"
"#2 · @Women_State · Мода и красотаДевочки, сегодня подборка casual образов к осени от российского бренда El.ka br. *Пост не является рекламным и отражает исключительно персональные предпочтения женской редакции канала. Все ссылки даем по собственной инициативе и доброй воле Давайте вместе выберем ниже самый классный образ","#2 · @favouriteWB24 · Мода и красотаНевероятно красивый классический комплект с брюками и жилеткой Он идеально подходит для официальных мероприятий, корпоративных вечеров и торжественных случаев. Наряд двойка классика - это не только образ для работы, праздника но и символ стиля и элегантности, который подчеркнет вашу индивидуальность. 221401647"
"#3 · @Women_State · Мода и красотаДевочки, сегодня подборка casual образов к осени от российского бренда El.ka br. *Пост не является рекламным и отражает исключительно персональные предпочтения женской редакции канала. Все ссылки даем по собственной инициативе и доброй воле Давайте вместе выберем ниже самый классный образ","#3 · @wb_black_fridayy · ПродажиБлузка нарядная с длинным рукавомВесна и лето - время, когда каждая девушка хочет выглядеть нарядно и стильно. Блузка женская поможет тебе создать образ, который подчеркнет твою красоту и элегантность. Идеально подойдет для офисного стиля или вечернего наряда. Материал блузки - вискоза, он приятный на ощупь и отличается хорошей прочностью. Благодаря большой размерной сетки, эти блузки подойдет для разных типов фигур. Рейтинг: 4.7 Артикул: 203048471Цена: 1180 ₽ССЫЛКА НА БЛУЗКУ"
"#4 · @yessshecan · ПсихологияИдеи для образа! Джинсы-бойфренды и свитшот. Создай уютный и стильный образ, сочетая джинсы-бойфренды с мягким свитшотом. Этот лук подойдет для прогулок по парку или встреч с друзьями. Юбка-миди и топ в полоску. Неотъемлемый элемент весеннего гардероба - юбка-миди. Сочетай её с топом в полоску для создания элегантного и модного образа. Легкая блуза и джинсы-скинни. Простой, но стильный образ, который подходит для любой ситуации. Добавь к нему яркие аксессуары или легкую куртку для завершения лука. Yes, she can…","#4 · @toniajensen · Мода и красотаАктуальный элемент гардероба — поло в полоскуСтилизуем этот элемент в стиле преппи с лоферами, шортами и выглядывающими носками, на контрасте с женственной юбкой или просто с джинсами и брюками для создания расслабленного образа"
"#5 · @lowfashion · Мода и красотаThe New York Times задается вопросом: что же такого притягательного в шарфах The rainbow Acne Studios, что их носит каждый второй модник и любитель тепла? Прочитала материал Мисти Уайт Сайделл про эти радужные аксессуары и пришла к такому же выводу: они просто веселые, добавляют акцент в образ, напоминают по hygge-эстетику и красивые. Необязательно иметь заветный шарф от бренда за парочку десятков долларов, но обзавестись инфан

In [12]:
# Освобождаем VRAM
for k in list(loaded_models.keys()):
    del loaded_models[k]
loaded_models.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Память освобождена.')


Память освобождена.
